In [3]:
import requests
import pandas as pd

# ============================================================
# 1. Descargar y cargar datos
# ============================================================

url = "https://raw.githubusercontent.com/VC2015/DMLonGitHub/master/penn_jae.dat"
destfile = "penn_jae.dat"

# Descargar archivo
response = requests.get(url)
with open(destfile, "wb") as f:
    f.write(response.content)

print("✔ Datos descargados correctamente")

# Leer archivo (delimitado por espacios)
df = pd.read_csv(destfile, sep=r"\s+", engine="python")

print("✔ Datos cargados correctamente")
print("Tamaño:", df.shape)

# Mostrar primeras filas
display(df.head())



✔ Datos descargados correctamente
✔ Datos cargados correctamente
Tamaño: (13913, 23)


,abdt,tg,inuidur1,inuidur2,female,black,hispanic,othrace,dep,q1,...,q5,q6,recall,agelt35,agegt54,durable,nondurable,lusd,husd,muld
0,10824,0,18,18,0,0,0,0,2,0,...,1,0,0,0,0,0,0,0,1,0
1,10635,2,7,3,0,0,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0
2,10551,5,18,6,1,0,0,0,0,0,...,0,0,1,0,1,0,0,0,0,0
3,10824,0,1,1,0,0,0,0,0,0,...,1,0,0,0,0,0,0,1,0,0
4,10747,0,27,27,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import LassoCV, LinearRegression   # 👈 ESTA LÍNEA ES CLAVE
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
import os


In [5]:
# =========================================================
# Cleaning and Set-up
# =========================================================

# --- Crear carpeta de salida si no existe ---
OUTPUT_DIR = "../output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 1. Cargar datos desde tu ruta local ---
file_path = r"C:\Users\Dafne\Documents\GitHub\CausalAI-Course\data\penn_jae.dat"


print("✅ Datos cargados correctamente")
print("Shape original:", df.shape)
print(df.head())

# --- 2. Filtrar sólo observaciones donde 'tg' == 0 o 4 ---
df = df[df['tg'].isin([0, 4])].copy()

# --- 3. Definir variable de tratamiento ---
df['T4'] = (df['tg'] == 4).astype(int)   # 1 si tg=4, 0 si tg=0

# --- 4. Definir variable de resultado (y) ---
df['y'] = np.log(df['inuidur1'])

# --- 5. Crear variables dummy para 'dep' ---
df = pd.get_dummies(df, columns=['dep'], prefix='dep')

# Mostrar qué dummies se generaron
print("Columnas dummies de 'dep':", [c for c in df.columns if c.startswith('dep_')])

# --- 6. Definir covariables (X) ---
x_vars = [
    'female','black','othrace',
    'dep_1','dep_2',
    'q2','q3','q4','q5','q6',
    'recall','agelt35','agegt54',
    'durable','nondurable','lusd','husd'
]

# Comprobar si falta alguna variable
missing = [v for v in x_vars if v not in df.columns]
if missing:
    print("⚠️ Variables faltantes:", missing)

# --- 7. Definir conjuntos finales ---
y = df['y']
d = df['T4']
X = df[x_vars]

print("\n✅ Preparación completa")
print("y:", y.shape)
print("d:", d.shape)
print("X:", X.shape)


✅ Datos cargados correctamente
Shape original: (13913, 23)
    abdt  tg  inuidur1  inuidur2  female  black  hispanic  othrace  dep  q1  \
0  10824   0        18        18       0      0         0        0    2   0   
1  10635   2         7         3       0      0         0        0    0   0   
2  10551   5        18         6       1      0         0        0    0   0   
3  10824   0         1         1       0      0         0        0    0   0   
4  10747   0        27        27       0      0         0        0    0   0   

   ...  q5  q6  recall  agelt35  agegt54  durable  nondurable  lusd  husd  \
0  ...   1   0       0        0        0        0           0     0     1   
1  ...   0   0       0        1        0        0           0     1     0   
2  ...   0   0       1        0        1        0           0     0     0   
3  ...   1   0       0        0        0        0           0     1     0   
4  ...   0   0       0        0        0        0           0     1     0   

   

In [6]:
# =========================================================
# Definir función DML (cross-fitting)
# =========================================================

def dml(y, d, X, ml_method="lasso", n_splits=2, random_state=42):
    """
    Implementa Debiased Machine Learning (DML)
    para el modelo parcialmente lineal con cross-fitting.
    
    Parámetros:
    ------------
    y : array-like
        Variable dependiente (outcome)
    d : array-like
        Variable de tratamiento
    X : DataFrame
        Covariables
    ml_method : str
        Método ML a usar: 'ols', 'lasso', 'rf', 'nn'
    n_splits : int
        Número de folds para cross-fitting
    """

    # Seleccionar modelo base
    if ml_method == "ols":
        model_y = LinearRegression()
        model_d = LinearRegression()
    elif ml_method == "lasso":
        model_y = LassoCV(cv=5, random_state=random_state)
        model_d = LassoCV(cv=5, random_state=random_state)
    elif ml_method == "rf":
        model_y = RandomForestRegressor(n_estimators=200, random_state=random_state)
        model_d = RandomForestRegressor(n_estimators=200, random_state=random_state)
    elif ml_method == "nn":
        model_y = MLPRegressor(hidden_layer_sizes=(50, 20), max_iter=1000, random_state=random_state)
        model_d = MLPRegressor(hidden_layer_sizes=(50, 20), max_iter=1000, random_state=random_state)
    else:
        raise ValueError("Método no reconocido: usa 'ols', 'lasso', 'rf', o 'nn'.")

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    n = len(y)

    # Vectores vacíos para residuos
    y_tilde = np.zeros(n)
    d_tilde = np.zeros(n)

    # Cross-fitting
    for train_idx, test_idx in kf.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        d_train, d_test = d.iloc[train_idx], d.iloc[test_idx]

        # Estimar modelos auxiliares
        model_y.fit(X_train, y_train)
        model_d.fit(X_train, d_train)

        # Predicciones fuera del fold
        y_hat = model_y.predict(X_test)
        d_hat = model_d.predict(X_test)

        # Calcular residuos
        y_tilde[test_idx] = y_test - y_hat
        d_tilde[test_idx] = d_test - d_hat

    # Estimar theta con regresión OLS sobre residuos
    theta_hat = np.sum(d_tilde * y_tilde) / np.sum(d_tilde ** 2)

    # Calcular error estándar (aprox.)
    resid = y_tilde - theta_hat * d_tilde
    sigma2 = np.mean(resid ** 2)
    se = np.sqrt(sigma2 / np.sum(d_tilde ** 2))

    return {"theta": theta_hat, "se": se, "method": ml_method}


In [7]:
# =========================================================
# Estimar y comparar modelos
# =========================================================

modelos = ["ols", "lasso", "rf", "nn"]
results = []

for m in modelos:
    print(f"Estimando con {m.upper()}...")
    r = dml(y, d, X, ml_method=m, n_splits=2)
    results.append(r)

# Crear tabla de resultados
results_df = pd.DataFrame(results)
results_df["t_stat"] = results_df["theta"] / results_df["se"]
results_df


Estimando con OLS...
Estimando con LASSO...
Estimando con RF...
Estimando con NN...


,theta,se,method,t_stat
0,-0.060529,0.035161,ols,-1.721486
1,-0.070380,0.035289,lasso,-1.994371
2,-0.026599,0.034564,rf,-0.769562
3,-0.019272,0.034140,nn,-0.564496


breve análisis del cuadro: 
Los cuatro métodos producen coeficientes negativos, lo que sugiere que el tratamiento T4 reduce la duración del desempleo.
Los efectos estimados varían entre -0.02 y -0.07, siendo el modelo LASSO el más fuerte en magnitud y significancia estadística.

## Selección del mejor modelo
Dado que el modelo LASSO presenta el mayor valor absoluto del t-stat (≈ −1.99), se selecciona como el modelo principal.
El estimador sugiere que el tratamiento T4 reduce la duración del desempleo en aproximadamente 7 % (exp(−0.07) − 1 ≈ −0.07).
Esto implica que participar en el programa asociado a tg = 4 tiene un impacto negativo, aunque marginalmente significativo, sobre la duración del desempleo.

In [8]:
def dml_no_cf(y, d, X, ml_method="lasso", random_state=42):
    """
    Implementación de DML sin cross-fitting.
    Entrena los modelos auxiliares en todo el dataset y predice en el mismo dataset.

    Parámetros:
    ------------
    y : array-like
        Variable dependiente
    d : array-like
        Variable de tratamiento
    X : DataFrame
        Covariables
    ml_method : str
        'ols', 'lasso', 'rf', 'nn'
    """

    # Seleccionar modelo base
    if ml_method == "ols":
        model_y = LinearRegression()
        model_d = LinearRegression()

    elif ml_method == "lasso":
        model_y = LassoCV(cv=5, random_state=random_state)
        model_d = LassoCV(cv=5, random_state=random_state)

    elif ml_method == "rf":
        model_y = RandomForestRegressor(n_estimators=200, random_state=random_state)
        model_d = RandomForestRegressor(n_estimators=200, random_state=random_state)

    elif ml_method == "nn":
        model_y = MLPRegressor(hidden_layer_sizes=(50, 20), 
                               max_iter=1000, random_state=random_state)
        model_d = MLPRegressor(hidden_layer_sizes=(50, 20), 
                               max_iter=1000, random_state=random_state)

    else:
        raise ValueError("Método no reconocido: usa 'ols', 'lasso', 'rf', o 'nn'.")

    # ===============================
    # ENTRENAR UNA SOLA VEZ (NO CF)
    # ===============================

    # Ajustar modelos usando TODO X
    model_y.fit(X, y)
    model_d.fit(X, d)

    # Predicción en el mismo conjunto
    y_hat = model_y.predict(X)
    d_hat = model_d.predict(X)

    # Residuos
    y_tilde = y - y_hat
    d_tilde = d - d_hat

    # ===============================
    # Estimar theta
    # ===============================
    theta_hat = np.sum(d_tilde * y_tilde) / np.sum(d_tilde ** 2)

    # Error estándar
    resid = y_tilde - theta_hat * d_tilde
    sigma2 = np.mean(resid ** 2)
    se = np.sqrt(sigma2 / np.sum(d_tilde ** 2))

    return {"theta": theta_hat, "se": se, "method": ml_method}


In [9]:
# =========================================================
# Estimar y comparar modelos (SIN CROSS-FITTING)
# =========================================================

modelos = ["ols", "lasso", "rf", "nn"]
results_no_cf = []

for m in modelos:
    print(f"Estimando con {m.upper()} (sin cross-fitting)...")
    
    # Llamada al estimador sin cross-fitting
    r = dml_no_cf(y, d, X, ml_method=m)
    
    results_no_cf.append(r)

# Crear tabla de resultados
results_no_cf_df = pd.DataFrame(results_no_cf)
results_no_cf_df["t_stat"] = results_no_cf_df["theta"] / results_no_cf_df["se"]

results_no_cf_df


Estimando con OLS (sin cross-fitting)...
Estimando con LASSO (sin cross-fitting)...
Estimando con RF (sin cross-fitting)...
Estimando con NN (sin cross-fitting)...


,theta,se,method,t_stat
0,-0.072576,0.035205,ols,-2.061533
1,-0.072853,0.035164,lasso,-2.071789
2,-0.080478,0.035237,rf,-2.283884
3,-0.074694,0.034738,nn,-2.150244


Los modelos sin cross-fitting muestran valores de 
𝜃
θ más grandes en magnitud y t-stats más altos.

Esto se debe a que el RMSE de las predicciones auxiliares es artificialmente bajo, porque los modelos predicen en el mismo conjunto donde fueron entrenados.

Los modelos con cross-fitting, aunque tienen mayor error de predicción, generan residuos verdaderamente out-of-sample, evitando correlación espuria entre errores.

Cross-fitting es esencial para obtener un estimador DML insesgado y válido estadísticamente.

✅ 1. What can you say about the RMSE for predicting y and d?

Without cross-fitting, the RMSE of predicting 
𝑦
y and 
𝑑
d will always be smaller.

With cross-fitting, the RMSE becomes larger.

✔ Why?

Because:

Without cross-fitting:
The same data are used for training and prediction.
→ The model overfits.
→ Predictions are artificially accurate.
→ RMSE looks better than it really is.

With cross-fitting:
Each observation is predicted using a model trained on a different fold (out-of-sample).
→ RMSE reflects true predictive performance.
→ Therefore it's higher but more realistic.

🔎 Conclusion:

RMSE is lower without cross-fitting only because of overfitting.
With cross-fitting, RMSE reflects true out-of-sample error.

✅ 2. Why is it that estimating with one function yields lower RMSE than another?

Because the function without cross-fitting evaluates predictions in-sample, where the model has already seen the data.

Models always look better on the data they were trained on.

✔ Key point:

Lower RMSE does NOT mean the model is better — it just means it was evaluated on the same data used for training.

The function with cross-fitting forces the model to predict unseen data, exposing true prediction error.

✅ 3. What problem would we have if we chose to estimate without cross-fitting?

This is the heart of DML.

❌ Problem: The estimator becomes biased.

Without cross-fitting:

The machine learning models overfit to noise in the data.



This breaks the key DML orthogonality condition:

𝐸
[
𝑑
~
 
𝜖
]
=
0.
E[
d
~
ϵ]=0.
🔎 Consequences:

The causal effect 
𝜃
θ is biased.

t-statistics look artificially strong.

Standard errors are underestimated.

You may believe you have a significant effect when you do not.

✔ Final summary:

Without cross-fitting, DML loses its robustness.
The causal estimate is biased because overfitting leaks noise into the residuals.